# Task 1 Demo: MNIST Classification + OOP

This notebook demonstrates the structure of Task 1, the shared classifier API, a smoke-test run, and several edge cases that are useful to explain during review.

## What this notebook covers

- the common wrapper API exposed by `MnistClassifier`;
- how feature shapes change for `rf`, `nn`, and `cnn`;
- a reproducible smoke test using the local `digits` fallback dataset;
- edge cases such as invalid algorithm selection and dataset source routing.

The assignment target dataset remains MNIST. The `digits` dataset is used here only because it is lightweight and works well for a quick demo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project_dir = Path.cwd()
if not (project_dir / "main.py").exists():
    project_dir = project_dir / "Task 1"

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print(f"Notebook root: {project_dir}")

In [ ]:
from main import prepare_features
from src.mnist_classifier import MnistClassifier
import numpy as np

for algorithm in ["rf", "nn", "cnn"]:
    classifier = MnistClassifier(algorithm)
    print(algorithm, "->", classifier.model.__class__.__name__)

In [ ]:
X_train = np.zeros((4, 28, 28), dtype="float32")
X_test = np.zeros((2, 28, 28), dtype="float32")

for algorithm in ["rf", "nn", "cnn"]:
    prepared_train, prepared_test = prepare_features(algorithm, X_train, X_test)
    print(
        algorithm,
        "train shape:",
        prepared_train.shape,
        "test shape:",
        prepared_test.shape,
    )

## Smoke test run

This command runs the Random Forest model on the local `digits` fallback dataset. It is intentionally small and reproducible, so it is convenient for demos and CI-like checks.

In [ ]:
command = [
    sys.executable,
    "main.py",
    "--algorithm",
    "rf",
    "--dataset-source",
    "digits",
    "--sample-size",
    "500",
]
result = subprocess.run(command, cwd=project_dir, check=True, capture_output=True, text=True)
print(result.stdout)

## Full MNIST commands

These are the main training commands used for the assignment target dataset:

```bash
python3 "Task 1/main.py" --algorithm rf --sample-size 5000
python3 "Task 1/main.py" --algorithm nn --epochs 3 --sample-size 10000
python3 "Task 1/main.py" --algorithm cnn --epochs 3 --sample-size 10000
```

The exact runtime depends on whether TensorFlow is available and whether the environment can access the MNIST dataset cache or download source.

## Edge cases

The most useful edge cases to mention in review are:

1. invalid algorithm names should fail clearly;
2. `digits` should bypass the TensorFlow/Keras path completely;
3. `rf`, `nn`, and `cnn` require different input shapes;
4. `keras` should raise if Keras loading fails, while `auto` can fall back to OpenML.


In [ ]:
try:
    MnistClassifier("svm")
except ValueError as exc:
    print("Invalid algorithm error:", exc)

In [ ]:
from main import load_mnist

X_train, X_test, y_train, y_test = load_mnist("digits")
print("Digits fallback shapes:")
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)